In [ ]:
pip install qiskit qiskit-machine-learning qiskit-aer

In [ ]:
pip install --upgrade qiskit qiskit-machine-learning qiskit-aer

After running this cell, please restart the runtime (`Runtime > Restart runtime...`) to ensure the updated packages are loaded correctly.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
def load_data(path):
    df = pd.read_csv(path)

    # Target mapping
    df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]
    df['label'] = df['koi_disposition'].map({
        'CONFIRMED': 1,
        'FALSE POSITIVE': 0
    })

    features = [
        'koi_period', 'koi_time0bk', 'koi_impact',
        'koi_duration', 'koi_depth', 'koi_model_snr',
        'koi_prad', 'koi_teq', 'koi_insol',
        'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag'
    ]

    df = df[features + ['label']].dropna()

    X = df[features].values
    y = df['label'].values

    return X, y

In [ ]:
def create_vqc(n_qubits=5):

    x = ParameterVector('x', n_qubits)
    theta = ParameterVector('θ', n_qubits)
    qc = QuantumCircuit(n_qubits)

    for i in range(n_qubits):
        qc.ry(x[i], i)

    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)

    for i in range(n_qubits):
        qc.ry(theta[i], i)

    return qc, x, theta

In [ ]:
from qiskit.quantum_info import SparsePauliOp

def build_qnn(qc, x, theta):

    estimator = StatevectorEstimator()
    observable = SparsePauliOp.from_list([("Z" + "I"*(qc.num_qubits-1), 1)])

    qnn = EstimatorQNN(
        circuit=qc,
        observables=observable,   # ← REQUIRED
        input_params=x,
        weight_params=theta,
        estimator=estimator
    )

    return qnn

In [ ]:
def extract_quantum_features(qnn, X, observable):
    weights = np.random.rand(qnn.num_weights)

    features = []

    for sample in X:
        output = qnn.forward(sample, weights)
        features.append(output)

    return np.array(features)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_model(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2
    )

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    model = Classifier(X.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    # Evaluation
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    preds = model(X_test_t).detach().numpy()
    preds = (preds > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)

    print("Accuracy:", acc)

    return model

In [ ]:
def preprocess(X, n_components=5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    X_reduced = pca.fit_transform(X_scaled)

    return X_reduced

In [ ]:
def main():

    X, y = load_data(path="/exoplanets.csv")

    X_reduced = preprocess(X)

    qc, x, theta = create_vqc(n_qubits=5)

    qnn, observable = build_qnn(qc, x, theta) # Receive observable

    X_quantum = extract_quantum_features(qnn, X_reduced, observable) # Pass observable

    model = train_model(X_quantum, y)


if __name__ == "__main__":
    main()

TypeError: cannot unpack non-iterable EstimatorQNN object